# Etudier la correlation avec Chi2

**Étape : Test d’indépendance du Khi² (χ²)**

**Objectif**  
Après la discrétisation des variables numériques, cette étape vise à :  
1. Vérifier la dépendance entre chaque variable explicative discrétisée et la variable cible (`loan_status`), afin d’identifier les variables réellement prédictives du défaut.  
2. Détecter la multi-colinéarité entre les variables explicatives discrétisées, c’est-à-dire les variables fortement corrélées entre elles, qui pourraient redonder l’information.

**Méthode utilisée : Test du χ² d’indépendance**  
Le test du Khi² d’indépendance permet de mesurer si deux variables catégorielles sont statistiquement indépendantes.

**Hypothèses du test :**  
- H₀ (hypothèse nulle) : les deux variables sont indépendantes  
- H₁ (alternative) : les deux variables sont dépendantes (corrélées)

**Principe :**  
Le test compare les fréquences observées dans le tableau de contingence à celles qui seraient attendues si les deux variables étaient indépendantes.

**Interprétation :**  
- Si p-value < 0.05 → on rejette H₀, les variables sont dépendantes (donc liées entre elles).  
- Si p-value ≥ 0.05 → on ne rejette pas H₀, les variables sont indépendantes.

**Application pratique :**  
1. Entre chaque variable discrétisée et la variable cible (`loan_status`)  
   → permet d’identifier les variables explicatives significatives.  
2. Entre les variables explicatives discrétisées elles-mêmes  
   → permet de repérer les variables redondantes pour réduire la multi-colinéarité du modèle.

**Résultat attendu :**  
À la fin de cette étape :  
- On obtient une liste des variables significativement liées au risque de défaut.  
- On élimine les variables trop corrélées entre elles, afin de conserver un modèle stable, explicable et sans redondance statistique.


In [29]:
import pandas as pd
from scipy.stats import chi2_contingency
import numpy as np

In [30]:
df_discretise = pd.read_csv("../data/output/df_discretise.csv",
                            sep = ",")
df_discretise.head()

,Unnamed: 0.1,Unnamed: 0,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,...,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,person_age_bin,person_income_bin,person_emp_length_bin,cb_person_cred_hist_length_bin,loan_amnt_bin,loan_percent_income_bin,loan_intent_bin
0,0,1,21,9600,OWN,5.000,EDUCATION,B,1000,11.140,...,0.100,N,2,person_age_Bin1,person_income_Bin1,person_emp_length_Bin4,cb_person_cred_hist_length_Bin1,loan_amnt_Bin1,loan_percent_income_Bin2,EDUCATION
1,1,2,25,9600,MORTGAGE,1.000,MEDICAL,C,5500,12.870,...,0.570,N,3,person_age_Bin3,person_income_Bin1,person_emp_length_Bin1,cb_person_cred_hist_length_Bin1,loan_amnt_Bin2,loan_percent_income_Bin5,HOMEIMPROVEMENT_MEDICAL
2,2,3,23,65500,RENT,4.000,MEDICAL,C,35000,15.230,...,0.530,N,2,person_age_Bin2,person_income_Bin4,person_emp_length_Bin3,cb_person_cred_hist_length_Bin1,loan_amnt_Bin5,loan_percent_income_Bin5,HOMEIMPROVEMENT_MEDICAL
3,3,4,24,54400,RENT,8.000,MEDICAL,C,35000,14.270,...,0.550,Y,4,person_age_Bin2,person_income_Bin4,person_emp_length_Bin4,cb_person_cred_hist_length_Bin1,loan_amnt_Bin5,loan_percent_income_Bin5,HOMEIMPROVEMENT_MEDICAL
4,4,5,21,9900,OWN,2.000,VENTURE,A,2500,7.140,...,0.250,N,2,person_age_Bin1,person_income_Bin1,person_emp_length_Bin2,cb_person_cred_hist_length_Bin1,loan_amnt_Bin1,loan_percent_income_Bin3,VENTURE


In [31]:
df_discretise.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29731 entries, 0 to 29730
Data columns (total 21 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Unnamed: 0.1                    29731 non-null  int64  
 1   Unnamed: 0                      29731 non-null  int64  
 2   person_age                      29731 non-null  int64  
 3   person_income                   29731 non-null  int64  
 4   person_home_ownership           29731 non-null  object 
 5   person_emp_length               29731 non-null  float64
 6   loan_intent                     29731 non-null  object 
 7   loan_grade                      29731 non-null  object 
 8   loan_amnt                       29731 non-null  int64  
 9   loan_int_rate                   29731 non-null  float64
 10  loan_status                     29731 non-null  int64  
 11  loan_percent_income             29731 non-null  float64
 12  cb_person_default_on_file       

In [37]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

pd.set_option('display.float_format', lambda x: '%.3f' % x)

# --- Définir les variables catégorielles et discrétisées ---
discretized_vars = [col for col in df_discretise.columns if col.endswith("_bin")]

# Ensemble des variables explicatives catégorielles (ajout manuel)
chi_vars = list(set(discretized_vars + ["person_home_ownership"]))

target = "loan_status"

# --- 1️⃣ Test du χ² entre chaque variable explicative et la cible ---
chi2_target_results = []

for var in chi_vars:
    contingency = pd.crosstab(df_discretise[var], df_discretise[target])
    chi2, p, dof, expected = chi2_contingency(contingency)
    chi2_target_results.append({
        "variable": var,
        "chi2": chi2,
        "p_value": p,
        "dependante_avec_cible": p < 0.05
    })

chi2_target_df = pd.DataFrame(chi2_target_results).sort_values("p_value")
print("=== Test χ² : Variables explicatives vs Cible ===")
print(chi2_target_df)

# --- 2️⃣ Test du χ² entre variables explicatives (corrélation entre elles) ---
n = len(chi_vars)
chi2_matrix = pd.DataFrame(np.ones((n, n)), columns=chi_vars, index=chi_vars)

for i, var1 in enumerate(chi_vars):
    for j, var2 in enumerate(chi_vars):
        if i < j:
            contingency = pd.crosstab(df_discretise[var1], df_discretise[var2])
            chi2, p, dof, expected = chi2_contingency(contingency)
            chi2_matrix.loc[var1, var2] = p
            chi2_matrix.loc[var2, var1] = p

print("\n=== Matrice de p-values du test χ² entre variables explicatives ===")
chi2_matrix


=== Test χ² : Variables explicatives vs Cible ===
                         variable     chi2  p_value  dependante_avec_cible
1               person_income_bin 2303.243    0.000                   True
7           person_home_ownership 1681.889    0.000                   True
6         loan_percent_income_bin 5310.276    0.000                   True
5                   loan_amnt_bin  690.821    0.000                   True
2                 loan_intent_bin  476.733    0.000                   True
3           person_emp_length_bin  313.710    0.000                   True
4                  person_age_bin   45.219    0.000                   True
0  cb_person_cred_hist_length_bin   16.725    0.002                   True

=== Matrice de p-values du test χ² entre variables explicatives ===


,cb_person_cred_hist_length_bin,person_income_bin,loan_intent_bin,person_emp_length_bin,person_age_bin,loan_amnt_bin,loan_percent_income_bin,person_home_ownership
cb_person_cred_hist_length_bin,1.000,0.000,0.000,0.000,0.000,0.000,0.248,0.000
person_income_bin,0.000,1.000,0.087,0.000,0.000,0.000,0.000,0.000
loan_intent_bin,0.000,0.087,1.000,0.000,0.000,0.473,0.305,0.000
person_emp_length_bin,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000
person_age_bin,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000
loan_amnt_bin,0.000,0.000,0.473,0.000,0.000,1.000,0.000,0.000
loan_percent_income_bin,0.248,0.000,0.305,0.000,0.000,0.000,1.000,0.000
person_home_ownership,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000


In [34]:
# Sélection des variables catégorielles non discrétisées
cat_vars = [
    col for col in df_discretise.select_dtypes(include=["object", "category"]).columns
    if not col.endswith("_bin")
]

# Affichage des effectifs
print("=== Effectifs des variables catégorielles non discrétisées ===\n")

for var in cat_vars:
    print(f"\n📊 {var} :")
    print(df_discretise[var].value_counts())


=== Effectifs des variables catégorielles non discrétisées ===


📊 person_home_ownership :
person_home_ownership
RENT        15585
MORTGAGE    11791
OWN          2257
OTHER          98
Name: count, dtype: int64

📊 loan_intent :
loan_intent
EDUCATION            5934
MEDICAL              5596
VENTURE              5186
PERSONAL             5035
DEBTCONSOLIDATION    4746
HOMEIMPROVEMENT      3234
Name: count, dtype: int64

📊 loan_grade :
loan_grade
A    9763
B    9500
C    5987
D    3345
E     866
F     211
G      59
Name: count, dtype: int64

📊 cb_person_default_on_file :
cb_person_default_on_file
N    24448
Y     5283
Name: count, dtype: int64
